# Lab 2 — Graph Construction & Query

**Goal:** Store extracted triples in a NetworkX graph and query it — neighbours, shortest paths, predicate search.

**Requires:** `pip install networkx matplotlib`

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from dotenv import load_dotenv
load_dotenv(override=True)
from knowledge_graph import KnowledgeGraph, Triple
import networkx as nx
print('Ready ✓')

## 1. Build a graph from text

In [ ]:
TEXT = """
Elon Musk founded SpaceX in 2002 and Tesla in 2003. SpaceX is headquartered in Hawthorne, California.
Tesla is headquartered in Austin, Texas. In 2022, Musk acquired Twitter and renamed it X.
Gwynne Shotwell is the President of SpaceX. Sam Altman is the CEO of OpenAI.
OpenAI is based in San Francisco. Microsoft invested in OpenAI in 2019.
California is a state in the United States. Texas is also a US state.
"""

kg = KnowledgeGraph()
triples = kg.add_text(TEXT)

print(f'Graph stats: {kg.stats()}')
print(f'\nAll {len(triples)} triples:')
for t in triples:
    print(f'  ({t.subject}) --[{t.predicate}]--> ({t.obj})')

## 2. Query neighbours

In [ ]:
print('What does Elon Musk connect to?')
for n in kg.neighbors('Elon Musk'):
    print(f'  {n["direction"]} [{n["predicate"]}] {n["entity"]}')

## 3. Find by predicate

In [ ]:
# Find all headquarters relationships
hq_pairs = kg.find_by_predicate('headquartered_in')
if not hq_pairs:
    # Try common variations
    for edge_pred in set(d['predicate'] for _,_,d in kg.graph.edges(data=True)):
        if 'headquarter' in edge_pred.lower() or 'located' in edge_pred.lower() or 'based' in edge_pred.lower():
            hq_pairs += kg.find_by_predicate(edge_pred)
            
print('Companies and their locations:')
for org, loc in hq_pairs:
    print(f'  {org} → {loc}')

## 4. Shortest path between entities

In [ ]:
# What is the connection between Gwynne Shotwell and California?
path = kg.shortest_path('Gwynne Shotwell', 'California')
if path:
    print('Path: ' + ' → '.join(path))
else:
    print('No path found. Try the graph nodes:', list(kg.graph.nodes())[:10])

## 5. Visualise the graph

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(14, 8))
    pos = nx.spring_layout(kg.graph, k=2, seed=42)
    nx.draw_networkx_nodes(kg.graph, pos, node_size=1500, node_color='lightblue', alpha=0.8)
    nx.draw_networkx_labels(kg.graph, pos, font_size=8)
    edge_labels = {(s, o): d['predicate'] for s, o, d in kg.graph.edges(data=True)}
    nx.draw_networkx_edges(kg.graph, pos, alpha=0.5, arrows=True, arrowsize=20)
    nx.draw_networkx_edge_labels(kg.graph, pos, edge_labels=edge_labels, font_size=6)
    plt.title('Knowledge Graph')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not installed — skipping visualisation')

## 6. Save and reload

In [ ]:
import tempfile, os
path = os.path.join(tempfile.gettempdir(), 'demo_kg.json')
kg.save(path)

kg2 = KnowledgeGraph.load(path)
print(f'Saved and reloaded: {kg2.stats()}')
print('Graphs match:', kg.stats() == kg2.stats())

## Exercise
Build a function `entity_importance(kg)` that ranks entities by their degree centrality (number of connections).  
The most important entity in a knowledge graph is typically the most connected one.  
Print the top 5 entities and explain why they rank highest.

In [ ]:
# Your code here
